# Pediatric EEG V2 — full-cohort experiment

Choose **Runtime → Change runtime type → T4 GPU → Runtime version 2026.07** for Python 3.12. Raw EDFs, signal caches, and expanded release clips stay on runtime disk. Only project artifacts are written under `MyDrive/pediatric-eeg-seizure-v2`; no other Drive folders are inspected.

All 23 unique individuals; frozen 15/4/4 split. This notebook is an orchestration layer over the repository, not an alternate implementation. Run in a GPU runtime. Source EEG stays on the runtime disk; checkpoints and reports persist in your Drive. No test-based model selection.

Research/educational prototype. Not for diagnosis or clinical decisions.

In [ ]:
from google.colab import drive
drive.mount("/content/drive", timeout_ms=600000)


In [ ]:
import os, subprocess, shutil, sys
from pathlib import Path
assert sys.version_info[:2] == (3, 12), "Select Colab runtime version 2026.07 (Python 3.12)"
def run(command):
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        if process.wait():
            raise RuntimeError(f"Command failed: {command}")
REPO = "https://github.com/iampenuel/pediatric-eeg-seizure-v2.git"
PROJECT = Path("/content/pediatric-eeg-seizure-v2")
if not PROJECT.exists():
    run(["git", "clone", REPO, str(PROJECT)])
os.chdir(PROJECT)
run(["git", "rev-parse", "HEAD"])
run(["nvidia-smi"])
print("Free runtime GiB:", round(shutil.disk_usage("/content").free / 2**30, 1))


In [ ]:
run([sys.executable, "-m", "pip", "install", "--no-cache-dir", "-e", ".[data,train,export,serve,test]"])


In [ ]:
run([sys.executable, "-m", "pytest", "-q"])


## Full-cohort run

The workflow prepares all eligible recordings, audits all 23 individuals, trains both models, freezes validation-only selection, evaluates both on held-out patients, and creates the demo bundle. Restart this cell after a runtime interruption: source downloads/cache are verified, training resumes from the previous completed epoch, and frozen runs are not retrained. Do not change the configuration after inspecting test outputs. Runtime GPU availability and duration are not guaranteed.

In [ ]:
DATA = "/content/chbmit-v2"
OUTPUT = "/content/drive/MyDrive/pediatric-eeg-seizure-v2/full-seed42"
run([sys.executable, "-u", "scripts/run_study.py", "--data", DATA, "--output", OUTPUT])


In [ ]:
import json
summary = json.loads((Path(OUTPUT) / "reports/summary.json").read_text())
for name, report in summary.items():
    print(name, "pooled:", report["pooled"], "patient macro:", report["patient_macro"])
print("Demo release:", Path(OUTPUT) / "demo.tar.gz")
